# Milestone 1: Popularity baseline

Starter notebook for milestone 1 of [Project - MovieLens Recommender](Project%20-%20MovieLens%20Recommender.md).

Cells marked **TODO** are yours to fill in; everything else runs as-is. Notebook is saved without outputs.

In [ ]:
import numpy as np, pandas as pd, pathlib, urllib.request, zipfile, io

DATA = pathlib.Path("ml-latest-small")
if not DATA.exists():
    # Or download manually from https://grouplens.org/datasets/movielens/ (ml-latest-small) and unzip here
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
ratings = pd.read_csv(DATA / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "links.csv")       # movieId, imdbId, tmdbId
ratings.head()

In [ ]:
# Leave-last-out split by time: for every user, the most recent positive is the test item,
# the one before it is validation. Positives = rating >= 4 (implicit feedback).
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]

# Contiguous ids so we can index embedding tables later
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(movies.movieId.unique()))}   # includes never-rated movies (cold items)
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

In [ ]:
def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HitRate@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (sampled protocol).
    score_fn(user_id, item_ids) -> array of scores. Switch to full ranking for final numbers,
    sampled metrics can reorder models (see Evaluating Recommenders)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum() + rng.integers(0, (s == s[0]).sum())   # 0 = first; ties broken at random (a constant scorer must score ~chance)
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

## 1. Most-popular scorer
Rank by number of training positives. This is the number every later model has to beat.

In [ ]:
pop = train.movieId.value_counts()
def pop_score(u, items):
    return pop.reindex(items).fillna(0).to_numpy()
# TODO: evaluate pop_score on val. Then try a random scorer; it should give HR@10 close to 0.10 with 99 negatives.
# evaluate(pop_score, val, train)

## 2. Variants (TODO)
- Popularity in the last N days only (time-decayed).
- Per-genre popularity: use the user's most-watched genre.
- Bayesian-average rating instead of count: which of count vs mean rating wins, and why?

In [ ]:
# TODO: implement at least one variant and compare in a small table (HR@10, NDCG@10)

## 3. Look at the output
Print the top 10 titles of your best baseline. Does it look like a sensible recommender or just a list of blockbusters?

In [ ]:
# TODO: movies.set_index('movieId').loc[pop.index[:10], 'title']